# 🚀 HPC-AI SDK: Production SQL Fine-tuning on Qwen3-8B for $1.03

**Goal:** Transform a general-purpose LLM into a specialized SQL Engineer using the **HPC-AI Fine-tuning SDK**.

**The Workflow:**
We write standard PyTorch-style training loops locally (on a laptop/Mac Mini), but the heavy gradient computation is offloaded to **HPC Cloud Clusters**. This gives us the control of a local script with the power of a data center.

**The Results (Production Run):**
I ran a full production fine-tune in the background to test the limits.
*   **Model:** Qwen3-8B (Instruct)
*   **Dataset:** 10,000 Text-to-SQL pairs
*   **Training Steps:** 300 (approx 1 epoch)
*   **Final Loss:** **0.028** (Converged perfectly)
*   **Total Cost:** **$1.03**

Let's build the script that made this happen.

## 1. Setup & Connection
First, we initialize the `ServiceClient`. This acts as the bridge between our local Python environment and the remote GPU cluster.

In [ ]:
import hpcai
from hpcai import types
from hpcai.cookbook.renderers import Qwen3InstructRenderer, Message, TrainOnWhat
from hpcai.cookbook.data import conversation_to_datum
import numpy as np
from datasets import load_dataset
import os
import time
import math

# Set API environment variables
os.environ["HPCAI_BASE_URL"] = "https://hpc-ai.com/finetunesdk"
os.environ["HPCAI_API_KEY"] = "YPUR API KEY"
service_client = hpcai.ServiceClient()
print("✓ ServiceClient initialized")


✓ ServiceClient initialized


## 2. Configuration: Demo vs. Production

**👇 PRO TIP:**
In the code block below, I have set up a **"Demo Config"** so we can run this live during the tutorial without waiting 50 minutes.

However, I have also included the **"Production Config"** (commented out). That is the exact configuration I used to achieve the **$1.03 result** on the 8B model. You can uncomment that block to reproduce the full enterprise run.

In [1]:
# --- 🟢 DEMO CONFIG (Fast Walkthrough) ---
# Model Configuration
MODEL_NAME = "Qwen/Qwen3-4B"
LORA_RANK = 32  # Corresponds to LORA_R in original train_lora.py

# Training Parameters
NUM_TRAIN_SAMPLES = 1000  # Use 1000 samples for quick validation
NUM_VAL_SAMPLES = 100  # Number of validation samples
BATCH_SIZE = 4
LEARNING_RATE = 5e-5  # Lower learning rate to prevent overfitting (original 2e-4 was too fast)
MAX_LENGTH = 512  # Corresponds to MAX_SEQ_LENGTH in original train_lora.py

# Training Steps
TRAIN_STEPS = 30
SAVE_EVERY = 20  # Save checkpoint every 20 steps
EVAL_EVERY = 10  # Evaluate validation set every 10 steps

# --- 🔴 PRODUCTION GRADE CONFIG (The $1.03 Run) ---
# Uncomment this block to run the full enterprise training
# MODEL_NAME = "Qwen/Qwen3-8B"  # Larger 8B model
# LORA_RANK = 32                # Higher rank for complex logic
# NUM_TRAIN_SAMPLES = 10000     # Full dataset for convergence
# BATCH_SIZE = 32               # Efficient batching
# LEARNING_RATE = 2e-4          # Higher LR for LoRA
# TRAIN_STEPS = 312             # ~1 Epoch
# --------------------------------------------------

print(f"Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  LoRA Rank: {LORA_RANK}")
print(f"  Training Samples: {NUM_TRAIN_SAMPLES}")
print(f"  Validation Samples: {NUM_VAL_SAMPLES}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Max Length: {MAX_LENGTH}")
print(f"  Training Steps: {TRAIN_STEPS}")
print(f"  Eval Interval: Every {EVAL_EVERY} steps")


Configuration:
  Model: Qwen/Qwen3-4B
  LoRA Rank: 32
  Training Samples: 1000
  Validation Samples: 100
  Batch Size: 4
  Learning Rate: 5e-05
  Max Length: 512
  Training Steps: 30
  Eval Interval: Every 10 steps


## 3. Allocate Remote Resources
When we create the client, the SDK automatically allocates high-performance compute resources on the cloud. We don't need to provision servers or install drivers manually.

In [84]:
# Create LoRA Training Client
training_client = service_client.create_lora_training_client(
    base_model=MODEL_NAME,
    rank=LORA_RANK,
    seed=42
)

print(f"✓ TrainingClient created")
print(f"  Model ID: {training_client.model_id}")

# Get tokenizer
tokenizer = training_client.get_tokenizer()
print(f"✓ Tokenizer retrieved: {type(tokenizer).__name__}")


✓ TrainingClient created
  Model ID: 279fddb7-1df6-4000-ac32-821fb94055ce
✓ Tokenizer retrieved: Qwen2TokenizerFast


## 4. Data Pipeline (ChatML Handling)

SQL generation requires strict formatting. We use the SDK's `Qwen3InstructRenderer` to automatically apply the correct ChatML templates (`<|im_start|>`, `<|im_end|>`). This ensures the model learns the exact structure required for inference.

In [57]:
from hpcai.cookbook.renderers import Qwen3InstructRenderer, Message, TrainOnWhat
from hpcai.cookbook.data import conversation_to_datum

# Create Qwen3 Renderer (Using Qwen3InstructRenderer for Qwen3 Instruct models)
# This ensures correct ChatML format: <|im_start|>role\ncontent<|im_end|>
renderer = Qwen3InstructRenderer(tokenizer)
print(f"✓ Renderer created: {type(renderer).__name__}")


def format_sql_messages(example) -> list[Message]:
    """Format dataset into SDK Message format"""
    question = example.get("question", "")
    context = example.get("context", "")
    answer = example.get("answer", "")
    
    return [
        Message(
            role="system",
            content="You are a SQL expert. Given a question and database schema, generate the correct SQL query."
        ),
        Message(
            role="user",
            content=f"### Question: {question}\n\n### Database Schema:\n{context}\n\nGenerate the SQL query to answer the question."
        ),
        Message(
            role="assistant",
            content=answer
        )
    ]


def row_to_datum(row) -> types.Datum:
    """Convert dataset row to SDK Datum format
    
    Uses Renderer to ensure:
    1. Correct ChatML template format
    2. Loss calculated only on assistant messages
    3. Correct handling of special tokens
    """
    messages = format_sql_messages(row)
    return conversation_to_datum(
        messages,
        renderer,
        max_length=MAX_LENGTH,
        train_on_what=TrainOnWhat.ALL_ASSISTANT_MESSAGES
    )


# Load training and validation sets (non-overlapping)
print("Loading dataset...")
train_dataset = load_dataset("b-mc2/sql-create-context", split=f"train[:{NUM_TRAIN_SAMPLES}]")
val_dataset = load_dataset("b-mc2/sql-create-context", split=f"train[{NUM_TRAIN_SAMPLES}:{NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES}]")
print(f"✓ Loaded {len(train_dataset)} training samples")
print(f"✓ Loaded {len(val_dataset)} validation samples")

# Pre-convert validation set to Datum format (avoids reconversion during each eval)
val_data = [row_to_datum(val_dataset[i]) for i in range(len(val_dataset))]
print(f"✓ Validation set pre-processed")

# Display a sample's tokens format (verify ChatML template correctness)
print("\nSample Example (ChatML Format):")
print("-" * 60)
sample_messages = format_sql_messages(train_dataset[0])
sample_tokens, sample_weights = renderer.build_supervised_example(
    sample_messages, 
    train_on_what=TrainOnWhat.ALL_ASSISTANT_MESSAGES
)
# Decode part of tokens to check format
decoded_sample = tokenizer.decode(sample_tokens[:80].tolist())
print(f"Token Prefix:\n{decoded_sample}...")
print(f"\nTotal tokens: {len(sample_tokens)}, Training tokens (weight=1): {sample_weights.sum().item():.0f}")
print("-" * 60)


✓ Renderer created: Qwen3InstructRenderer
Loading dataset...
✓ Loaded 1000 training samples
✓ Loaded 100 validation samples
✓ Validation set pre-processed

Sample Example (ChatML Format):
------------------------------------------------------------
Token Prefix:
<|im_start|>system
You are a SQL expert. Given a question and database schema, generate the correct SQL query.<|im_end|>
<|im_start|>user
### Question: How many heads of the departments are older than 56 ?

### Database Schema:
CREATE TABLE head (age INTEGER)

Generate the SQL query to answer the question.<|im_end|>
<|im_start|>assistant
SELECT COUNT(*) FROM head WHERE age > 56<|im_end|>...

Total tokens: 80, Training tokens (weight=1): 12
------------------------------------------------------------


## 5. Training Loop (Local Control, Cloud Execution)

**This is the "Magic" of the SDK.**

We are writing a standard Python loop. We control the data injection, the logging, and the scheduler locally.
*   **Local:** Data loading, Logic, Print statements.
*   **Cloud:** When `training_client.forward_backward()` is called, the batch is sent to the cluster, gradients are computed, and metrics are returned.

*Note: The code below includes a cosine decay scheduler (commented out) which was used for the production run.*

In [ ]:
import math

n_batches = len(train_dataset) // BATCH_SIZE
target_steps = min(n_batches, TRAIN_STEPS)

# Track training history
train_losses = []
val_losses = []
best_val_loss = float('inf')

print(f"Starting Training...")
print(f"  Total Samples: {len(train_dataset)}")
print(f"  Batches: {n_batches}")
print(f"  Target Steps: {target_steps}")
print(f"  Eval Interval: Every {EVAL_EVERY} steps")
print()

for step in range(target_steps):
    start_time = time.time()
    
    # Prepare batch
    batch_start = step * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, len(train_dataset))
    
    # Convert to Datum format using row_to_datum (with correct ChatML template)
    batch = [row_to_datum(train_dataset[i]) for i in range(batch_start, batch_end)]
    
    # Forward and Backward pass
    fwd_bwd = training_client.forward_backward(batch, "cross_entropy")
    
    # Optimizer update (using cosine learning rate decay)
    lr = LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * step / target_steps))


    # --- SCHEDULER LOGIC (Uncomment for Production) ---
    # WARMUP_STEPS = 30
    # if step < WARMUP_STEPS:
    #     lr = LEARNING_RATE * (step + 1) / WARMUP_STEPS
    # else:
    #     progress = (step - WARMUP_STEPS) / (TRAIN_STEPS - WARMUP_STEPS)
    #     lr = LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * progress))
    # --------------------------------------------------


    optim = training_client.optim_step(types.AdamParams(learning_rate=lr))
    
    # Get results
    result = fwd_bwd.result()
    optim.result()
    
    loss = result.metrics.get("loss:mean", 0.0)
    train_losses.append(loss)
    elapsed = time.time() - start_time
    
    print(f"Step {step+1}/{target_steps} | Train Loss: {loss:.4f} | LR: {lr:.2e} | Time: {elapsed:.2f}s")
    
    # Evaluate validation set periodically
    if (step + 1) % EVAL_EVERY == 0:
        val_result = training_client.forward(val_data, "cross_entropy").result()
        val_loss = val_result.metrics['loss:mean']
        val_losses.append((step + 1, val_loss))
        
        # Check if it's the best validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"  📊 Val Loss: {val_loss:.4f} ⭐ (new best)")
        else:
            print(f"  📊 Val Loss: {val_loss:.4f}")
    
    # Save checkpoint periodically
    if (step + 1) % SAVE_EVERY == 0:
        save_result = training_client.save_state(f"step_{step+1}_state").result()
        print(f"  ✓ Checkpoint saved: {save_result.path}")
        sampler_weights = training_client.save_weights_for_sampler(f"step_{step+1}_sampler").result()
        print(f"✓ Sampler weights saved: {sampler_weights.path}")

print("\n✓ Training Completed!")
print(f"  Final Train Loss: {train_losses[-1]:.4f}")
print(f"  Best Val Loss: {best_val_loss:.4f}")

# Display validation loss history
print("\nValidation Loss History:")
for step_num, vloss in val_losses:
    marker = " ⭐" if vloss == best_val_loss else ""
    print(f"  Step {step_num}: {vloss:.4f}{marker}")


Starting Training...
  Total Samples: 1000
  Batches: 250
  Target Steps: 30
  Eval Interval: Every 15 steps

Step 1/30 | Train Loss: 4.0312 | LR: 5.00e-05 | Time: 2.93s
Step 2/30 | Train Loss: 1.9531 | LR: 4.99e-05 | Time: 2.84s
Step 3/30 | Train Loss: 1.7188 | LR: 4.95e-05 | Time: 2.82s
Step 4/30 | Train Loss: 0.8398 | LR: 4.88e-05 | Time: 2.90s
Step 5/30 | Train Loss: 1.0859 | LR: 4.78e-05 | Time: 1.87s
Step 6/30 | Train Loss: 0.1172 | LR: 4.67e-05 | Time: 2.85s
Step 7/30 | Train Loss: 0.4922 | LR: 4.52e-05 | Time: 2.07s
Step 8/30 | Train Loss: 0.3848 | LR: 4.36e-05 | Time: 1.84s
Step 9/30 | Train Loss: 0.3164 | LR: 4.17e-05 | Time: 1.92s
Step 10/30 | Train Loss: 0.3652 | LR: 3.97e-05 | Time: 2.02s
Step 11/30 | Train Loss: 0.3867 | LR: 3.75e-05 | Time: 1.89s
Step 12/30 | Train Loss: 0.2988 | LR: 3.52e-05 | Time: 2.84s
Step 13/30 | Train Loss: 0.2539 | LR: 3.27e-05 | Time: 2.13s
Step 14/30 | Train Loss: 0.1641 | LR: 3.02e-05 | Time: 1.96s
Step 15/30 | Train Loss: 0.1709 | LR: 2.76e-0

## 6. Save Checkpoints & Adapter
We save the LoRA adapter weights. Since we "own" these weights, we can download them, merge them, or serve them anywhere.

In [59]:
# Save training state (includes optimizer state, can be used to resume training)
final_state = training_client.save_state("sql_adapter_final").result()
print(f"✓ Training state saved: {final_state.path}")

# Save inference weights (only weights, for inference, smaller file size)
sampler_weights = training_client.save_weights_for_sampler("sql_adapter_sampler").result()
print(f"✓ Inference weights saved: {sampler_weights.path}")

print("\nSaved Checkpoint Paths:")
print(f"  Training State: {final_state.path}")
print(f"  Inference Weights: {sampler_weights.path}")


✓ Training state saved: hpcai://89eaa796-24cb-4591-b3df-9a990c608958/training/sql_adapter_final
✓ Inference weights saved: hpcai://89eaa796-24cb-4591-b3df-9a990c608958/sampler/sql_adapter_sampler

Saved Checkpoint Paths:
  Training State: hpcai://89eaa796-24cb-4591-b3df-9a990c608958/training/sql_adapter_final
  Inference Weights: hpcai://89eaa796-24cb-4591-b3df-9a990c608958/sampler/sql_adapter_sampler


## 7. Evaluate Model (Sanity Check)
Before unloading the model, we run a few validation samples through `forward()` to see how much the loss has improved compared to the initial state.

In [60]:
# Test cases (Consistent with evaluate_lora.py)
TEST_CASES = [
    {
        "name": "Simple Query - Count",
        "question": "How many heads of the departments are older than 56?",
        "context": "CREATE TABLE head (age INTEGER)",
        "answer": "SELECT COUNT(*) FROM head WHERE age > 56",  # Use answer field to match format_sql_messages
    },
    {
        "name": "Conditional Query - Filtering",
        "question": "Find all products with price greater than 100",
        "context": "CREATE TABLE products (id INT, name VARCHAR(100), price DECIMAL(10,2))",
        "answer": "SELECT * FROM products WHERE price > 100",
    },
    {
        "name": "Aggregate Query - Sum",
        "question": "What is the total sales amount?",
        "context": "CREATE TABLE sales (id INT, product VARCHAR(100), amount DECIMAL(10,2))",
        "answer": "SELECT SUM(amount) FROM sales",
    },
    {
        "name": "Join Query - Table Association",
        "question": "List all employees and their department names",
        "context": "CREATE TABLE employees (id INT, name VARCHAR(100), dept_id INT); CREATE TABLE departments (id INT, dept_name VARCHAR(100))",
        "answer": "SELECT e.name, d.dept_name FROM employees e JOIN departments d ON e.dept_id = d.id",
    },
    {
        "name": "Grouping Query - GROUP BY",
        "question": "How many orders does each customer have?",
        "context": "CREATE TABLE orders (order_id INT, customer_id INT, order_date DATE)",
        "answer": "SELECT customer_id, COUNT(*) FROM orders GROUP BY customer_id",
    },
]

print("Evaluating Test Sample Loss:")
print("=" * 70)

test_losses = []
for i, test in enumerate(TEST_CASES, 1):
    # Convert to Datum using row_to_datum (using correct ChatML format)
    datum = row_to_datum(test)
    
    # Calculate loss
    result = training_client.forward([datum], "cross_entropy").result()
    loss = result.metrics['loss:mean']
    test_losses.append(loss)
    
    print(f"\nTest {i}: {test['name']}")
    print(f"  Question: {test['question']}")
    print(f"  Expected: {test['answer']}")
    print(f"  Loss: {loss:.4f}")

print("\n" + "=" * 70)
print(f"Average Loss: {np.mean(test_losses):.4f}")
print(f"Min Loss: {np.min(test_losses):.4f}")
print(f"Max Loss: {np.max(test_losses):.4f}")


Evaluating Test Sample Loss:

Test 1: Simple Query - Count
  Question: How many heads of the departments are older than 56?
  Expected: SELECT COUNT(*) FROM head WHERE age > 56
  Loss: 0.0337

Test 2: Conditional Query - Filtering
  Question: Find all products with price greater than 100
  Expected: SELECT * FROM products WHERE price > 100
  Loss: 0.0369

Test 3: Aggregate Query - Sum
  Question: What is the total sales amount?
  Expected: SELECT SUM(amount) FROM sales
  Loss: 0.0142

Test 4: Join Query - Table Association
  Question: List all employees and their department names
  Expected: SELECT e.name, d.dept_name FROM employees e JOIN departments d ON e.dept_id = d.id
  Loss: 0.1455

Test 5: Grouping Query - GROUP BY
  Question: How many orders does each customer have?
  Expected: SELECT customer_id, COUNT(*) FROM orders GROUP BY customer_id
  Loss: 0.2461

Average Loss: 0.0953
Min Loss: 0.0142
Max Loss: 0.2461


## 8. Run Inference
Now we switch to generation mode. We will create a `SamplingClient` using the trained weights to test if the model can generate valid SQL queries for new, unseen questions.

In [67]:
# Import SDK Renderer to build prompt format consistent with training
from hpcai.cookbook.renderers import Qwen3InstructRenderer, Message

# Create Qwen3 Renderer (Consistent with training format)
renderer = Qwen3InstructRenderer(tokenizer)
print(f"✓ Renderer created: {type(renderer).__name__}")

# Get stop token (to stop after generating SQL)
stop_tokens = renderer.get_stop_sequences()
print(f"✓ Stop tokens: {stop_tokens}")
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen3-4B")

def build_sql_prompt(question: str, context: str = "") -> types.ModelInput:
    """Build SQL prompt consistent with training format (ChatML format)
    
    Args:
        question: SQL question
        context: Database schema (CREATE TABLE statement)
    
    Returns:
        ModelInput: Formatted prompt
    """
    messages = [
        Message(
            role="system",
            content="You are a SQL expert. Given a question and database schema, generate the correct SQL query."
        ),
        Message(
            role="user",
            content=f"### Question: {question}\n\n### Database Schema:\n{context}\n\nGenerate the SQL query to answer the question."
        )
    ]
    return renderer.build_generation_prompt(messages)

# SQL related test cases (including questions and database schema, matching training format)
sql_test_cases = [
    {
        "question": "What is the total sales amount?",
        "context": "CREATE TABLE sales (id INT, product VARCHAR(100), amount DECIMAL(10,2), sale_date DATE)"
    },
    {
        "question": "Find all products with price greater than 100",
        "context": "CREATE TABLE products (id INT, name VARCHAR(100), price DECIMAL(10,2), category VARCHAR(50))"
    },
    {
        "question": "How many orders does each customer have?",
        "context": "CREATE TABLE orders (order_id INT, customer_id INT, order_date DATE, total DECIMAL(10,2))"
    }
]

print("SQL Test Cases:")
for i, case in enumerate(sql_test_cases, 1):
    print(f"  {i}. {case['question']}")
    print(f"     Schema: {case['context'][:60]}...")

✓ Renderer created: Qwen3InstructRenderer
✓ Stop tokens: [151645]
SQL Test Cases:
  1. What is the total sales amount?
     Schema: CREATE TABLE sales (id INT, product VARCHAR(100), amount DEC...
  2. Find all products with price greater than 100
     Schema: CREATE TABLE products (id INT, name VARCHAR(100), price DECI...
  3. How many orders does each customer have?
     Schema: CREATE TABLE orders (order_id INT, customer_id INT, order_da...


In [68]:
# Create SamplingClient for Base Model
base_sampling_client = service_client.create_sampling_client(
    base_model=MODEL_NAME
)
print(f"✓ Base Model SamplingClient created")
print(f"  Base Model: {MODEL_NAME}")
print(f"  Model Path: {base_sampling_client.model_path}")




✓ Base Model SamplingClient created
  Base Model: Qwen/Qwen3-4B
  Model Path: None


In [69]:
lora_sampling_client = service_client.create_sampling_client(
    base_model=MODEL_NAME,
    model_path=sampler_weights.path
)
print(f"✓ LoRA Model SamplingClient created")
print(f"  Base Model: {MODEL_NAME}")
print(f"  Model Path: {lora_sampling_client.model_path}")

✓ LoRA Model SamplingClient created
  Base Model: Qwen/Qwen3-4B
  Model Path: hpcai://89eaa796-24cb-4591-b3df-9a990c608958/sampler/sql_adapter_sampler


In [70]:
# Test the first SQL case
test_case = sql_test_cases[0]
print(f"\n=== Testing SQL Prompt ===")
print(f"Question: {test_case['question']}")
print(f"Context: {test_case['context']}")

# Use build_sql_prompt to build a prompt consistent with the training format (ChatML format)
sql_prompt_input = build_sql_prompt(
    question=test_case['question'],
    context=test_case['context']
)
print(f"✓ Prompt built (Using ChatML format)")

# Configure sampling parameters (Use stop token to stop after generating SQL)
sql_sampling_params = types.SamplingParams(
    max_tokens=128,
    temperature=0.1,  # Lower temperature for more deterministic SQL output
    stop=stop_tokens  # Use <|im_end|> token to stop generation
)
print(f"✓ Sampling Params: max_tokens={sql_sampling_params.max_tokens}, temperature={sql_sampling_params.temperature}, stop={stop_tokens}")




=== Testing SQL Prompt ===
Question: What is the total sales amount?
Context: CREATE TABLE sales (id INT, product VARCHAR(100), amount DECIMAL(10,2), sale_date DATE)
✓ Prompt built (Using ChatML format)
✓ Sampling Params: max_tokens=128, temperature=0.1, stop=[151645]


In [74]:
# Execute Base Model Sampling
print("\n--- Executing Base Model Sampling ---")
start_time = time.time()

future = base_sampling_client.sample(
    prompt=sql_prompt_input,
    num_samples=1,
    sampling_params=sql_sampling_params
)

result = future.result()
elapsed = time.time() - start_time

print(f"✓ Sampling completed (Time: {elapsed:.2f}s)")
print(f"  Generated sequences: {len(result.sequences)}")
print(f"  Generated tokens: {len(result.sequences[0].tokens)}")
print(f"  Stop reason: {result.sequences[0].stop_reason}")
# Decode and display results
generated_tokens = result.sequences[0].tokens
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n=== Base Model Generation Result ===")
print(f"Question: {test_case['question']}")
print(f"Generated: {generated_text}")


--- Executing Base Model Sampling ---


Sampling is paused for None. Reason: request is in queue, waiting for available resources


✓ Sampling completed (Time: 4.37s)
  Generated sequences: 1
  Generated tokens: 128
  Stop reason: length

=== Base Model Generation Result ===
Question: What is the total sales amount?
Generated: <think>
Okay, the user is asking for the total sales amount from the sales table. Let me think about how to approach this.

First, I need to sum up all the amounts in the sales table. The table has an 'amount' column which is of type DECIMAL. So the main operation here is to use the SUM() function on the amount column.

The schema has a sales table with columns id, product, amount, and sale_date. The question doesn't specify any filters, so I don't need to add any WHERE clause. 

Wait, but maybe they want the total across all sales. So the SQL query


In [75]:
lora_sampling_client = service_client.create_sampling_client(
    base_model=MODEL_NAME,
    model_path=sampler_weights.path
)
print(f"✓ LoRA Model SamplingClient created")
print(f"  Base Model: {MODEL_NAME}")
print(f"  Model Path: {lora_sampling_client.model_path}")
# Execute LoRA Model Sampling
print("\n--- Executing LoRA Model Sampling ---")
start_time = time.time()

future = lora_sampling_client.sample(
    prompt=sql_prompt_input,
    num_samples=1,
    sampling_params=sql_sampling_params
)

result = future.result()
elapsed = time.time() - start_time

print(f"✓ Sampling completed (Time: {elapsed:.2f}s)")
print(f"  Generated sequences: {len(result.sequences)}")
print(f"  Generated tokens: {len(result.sequences[0].tokens)}")
print(f"  Stop reason: {result.sequences[0].stop_reason}")
# Decode and display results
lora_generated_tokens = result.sequences[0].tokens
lora_generated_text = tokenizer.decode(lora_generated_tokens, skip_special_tokens=True)

print("\n=== LoRA Model Generation Result ===")
print(f"Question: {test_case['question']}")
print(f"Generated: {lora_generated_text}")

✓ LoRA Model SamplingClient created
  Base Model: Qwen/Qwen3-4B
  Model Path: hpcai://89eaa796-24cb-4591-b3df-9a990c608958/sampler/sql_adapter_sampler

--- Executing LoRA Model Sampling ---


Sampling is paused for hpcai://89eaa796-24cb-4591-b3df-9a990c608958/sampler/sql_adapter_sampler. Reason: request is in queue, waiting for available resources


✓ Sampling completed (Time: 1.84s)
  Generated sequences: 1
  Generated tokens: 7
  Stop reason: stop

=== LoRA Model Generation Result ===
Question: What is the total sales amount?
Generated: SELECT SUM(amount) FROM sales


## 9. Cleanup Resources
Always unload the model when finished to stop the billing meter!

In [85]:
training_client.unload_model().result()
print("✓ Model unloaded")


✓ Model unloaded


## Appendix

### Resuming Training from Checkpoint

If you need to resume training from a saved checkpoint, use the following code:

```python
# Resume training client from checkpoint
checkpoint_path = "hpcai://<model_id>/weights/sql_adapter_final"
training_client = service_client.create_training_client_from_state(checkpoint_path)

# Continue training...
```

### Downloading Checkpoints Locally

The `download_checkpoint_archive` method returns `bytes` data and **does not automatically save to a file**. You need to write it manually:

```python
# Create RestClient
rest_client = service_client.create_rest_client()

# Download checkpoint (returns bytes)
checkpoint_path = "hpcai://<model_id>/weights/sql_adapter_final"
archive = rest_client.download_checkpoint_archive_by_checkpoint_path(checkpoint_path).result()

# Manually save to local file (defaults to current working directory)
save_path = "./sql_adapter_final.tar.gz"
with open(save_path, "wb") as f:
    f.write(archive)

print(f"✓ Checkpoint downloaded to: {save_path}")
```

**Important Notes**:

1. **File Format**: Although the filename is `.tar.gz`, the actual content is **Zstandard (zstd)** compressed format, not standard gzip.
2. **Decompression**: macOS Archive Utility cannot decompress directly. Use command line tools:
   ```bash
   # Step 1: Decompress zstd format
   zstd -d sql_adapter_sampler.tar.gz -o sql_adapter_sampler.tar
   
   # Step 2: Extract tar file
   mkdir -p sql_adapter_sampler
   tar -xf sql_adapter_sampler.tar -C sql_adapter_sampler
   ```
3. **Install zstd**: If zstd is not installed, install via Homebrew:
   ```bash
   brew install zstd
   ```


In [ ]:
# Example: Download checkpoint to local
# Replace with actual checkpoint path (from Cell 6)
checkpoint_path = "hpcai://<model_id>/weights/sql_adapter_final"  # Replace with actual path

# Create RestClient
rest_client = service_client.create_rest_client()

# Download checkpoint (returns bytes)
print(f"Downloading checkpoint: {checkpoint_path}")
archive = rest_client.download_checkpoint_archive_by_checkpoint_path(checkpoint_path).result()

# Save to local file (saved in current working directory)
save_path = "./sql_adapter_final.tar.gz"
with open(save_path, "wb") as f:
    f.write(archive)

print(f"✓ Checkpoint downloaded to: {save_path}")
print(f"  File size: {len(archive) / 1024 / 1024:.2f} MB")


In [ ]:
# Decompress zstd format checkpoint in Python
import subprocess
import os

# Checkpoint file path (from download code above)
archive_path = "./sql_adapter_sampler.tar.gz"
output_dir = "./sql_adapter_sampler"

# Check if zstd is installed
try:
    subprocess.run(["zstd", "--version"], capture_output=True, check=True)
    print("✓ zstd is installed")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("✗ zstd not found, please install first: brew install zstd")
    raise

# Step 1: Decompress zstd format
tar_path = archive_path.replace(".tar.gz", ".tar")
print(f"Decompressing zstd format: {archive_path} -> {tar_path}")
subprocess.run(["zstd", "-d", archive_path, "-o", tar_path], check=True)
print("✓ zstd decompression completed")

# Step 2: Extract tar file
os.makedirs(output_dir, exist_ok=True)
print(f"Extracting tar file: {tar_path} -> {output_dir}/")
subprocess.run(["tar", "-xf", tar_path, "-C", output_dir], check=True)
print(f"✓ Extraction completed! Files are in {output_dir}/ directory")

# List extracted files
print("\nExtracted files:")
for f in os.listdir(output_dir):
    fpath = os.path.join(output_dir, f)
    size = os.path.getsize(fpath)
    if size > 1024 * 1024:
        print(f"  - {f} ({size / 1024 / 1024:.2f} MB)")
    else:
        print(f"  - {f} ({size / 1024:.2f} KB)")


In [ ]:
# 1. Create RestClient
rest_client = service_client.create_rest_client()

# 2. Download checkpoint (returns bytes)
# archive = rest_client.download_checkpoint_archive_by_checkpoint_path(
#     "hpcai://.../sampler_weights/sql_adapter_sampler_sampler"
# ).result()

# 3. Manually save to file
# save_path = "./sql_adapter_sampler.tar.gz"
# with open(save_path, "wb") as f:
#     f.write(archive)

# print(f"✓ Checkpoint downloaded to: {save_path}")